# 2. Data Collection — Web Scraping (Wikipedia)

Scrape Falcon 9 launch records from Wikipedia using BeautifulSoup.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata
import re

## 2.1 Helper Functions

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''
    for x in table_cells.strings:
        out += x.strip() + ' '
    return out.strip()

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize('NFKD', table_cells.text).strip()
    if mass:
        mass.find('kg')
        new_mass = mass[0:mass.find('kg')+2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not(column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name


## 2.2 Fetch Wikipedia Page

In [ ]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&action=raw'
response = requests.get(static_url)
print('Status:', response.status_code)

html = requests.get('https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches')
soup = BeautifulSoup(html.text, 'html.parser')
print('Page fetched successfully')


## 2.3 Extract Column Headers

In [ ]:
# Find all tables with wikitable class
html_tables = soup.find_all('table', 'wikitable')
print('Number of tables found:', len(html_tables))

# Extract headers from first launch table
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name and len(name) > 0:
        column_names.append(name)

print('Columns:', column_names)


## 2.4 Parse All Launch Tables

In [ ]:
launch_dict = dict.fromkeys(column_names)
for key in launch_dict:
    launch_dict[key] = []

extracted_row = 0
extracted_df  = pd.DataFrame()

for table in html_tables:
    for row in table.find_all('tr'):
        col = row.find_all('td')
        if len(col) == 9:
            Flight_No        = col[0].string
            Date_time        = date_time(col[1])
            Version_Booster  = booster_version(col[2])
            Launch_site      = col[3].a.string if col[3].a else col[3].string
            Payload          = col[4].a.string if col[4].a else col[4].string
            Payload_mass     = get_mass(col[5])
            Orbit            = col[6].a.string if col[6].a else col[6].string
            Customer         = col[7].a.string if col[7].a else col[7].string
            Launch_outcome   = landing_status(col[8])

            if Flight_No and Flight_No.strip().isdigit():
                extracted_row += 1
                launch_dict['Flight No.'].append(Flight_No)
                launch_dict['Launch site'].append(Launch_site)
                launch_dict['Payload'].append(Payload)
                launch_dict['Payload mass'].append(Payload_mass)
                launch_dict['Orbit'].append(Orbit)
                launch_dict['Customer'].append(Customer)
                launch_dict['Launch outcome'].append(Launch_outcome)
                launch_dict['Version Booster'].append(Version_Booster)
                launch_dict['Booster landing'].append(None)  # placeholder
                launch_dict['Date'].append(Date_time[0])
                launch_dict['Time'].append(Date_time[1] if len(Date_time)>1 else None)

print('Extracted rows:', extracted_row)


## 2.5 Save DataFrame

In [ ]:
df = pd.DataFrame(launch_dict)
print(df.shape)
df.head(10)


In [ ]:
df.to_csv('../data/spacex_web_scraped.csv', index=False)
print('Saved to spacex_web_scraped.csv')
